# Phase 1: Data Acquisition & Exploration
### AI-Based Energy Anomaly Detector
**Project Goal**: Flag wasteful/abnormal electricity usage in homes or buildings near real-time  
**SDG**: 7 (Affordable & Clean Energy), 12 (Responsible Consumption & Production)  
**Program**: 1M1B AI for Sustainability (IBM SkillsBuild & AICTE)

---
### Key Dataset Facts
- **Format**: Semicolon-separated (`;`)
- **Columns**: `Date`, `Time`, `Global_active_power`, `Global_reactive_power`, `Voltage`, `Global_intensity`, `Sub_metering_1`, `Sub_metering_2`, `Sub_metering_3`
- **Main Signal**: `Global_active_power` (in kilowatts) — total household power draw per minute
- **Missing Values**: Encoded as `?` strings
- **Scope**: Dec 2006 – Nov 2010, one household, minute-level readings (~2.07 million records)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

## 1. Load Raw Dataset
We specify `sep=';'`, `na_values='?'`, and parse the explicit datetime format for high performance.

In [ ]:
raw_path = Path('../data/raw/household_power_consumption.txt')
if not raw_path.exists():
    raw_path = Path('data/raw/household_power_consumption.txt')

print(f"Reading data from: {raw_path}")
df = pd.read_csv(
    raw_path,
    sep=';',
    na_values='?',
    low_memory=False
)
print("Initial raw shape:", df.shape)
df.head()

## 2. Datetime Parsing & Index Construction
Combine `Date` + `Time` into a single `datetime` column, sort chronologically, and set as the DataFrame index.

In [ ]:
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df.drop(columns=['Date', 'Time'], inplace=True)
df.set_index('datetime', inplace=True)
df.sort_index(inplace=True)

# Ensure all numeric columns are strictly float64
numeric_cols = [
    'Global_active_power', 'Global_reactive_power', 'Voltage',
    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Span: {df.index.min()} to {df.index.max()}")
print(f"Total records: {len(df):,}")
df.head()

## 3. Data Integrity & Descriptive Statistics
Verify there are no `'?'` strings remaining and inspect distribution of missing values and consumption stats.

In [ ]:
print("=== Missing Values Summary ===")
missing_counts = df.isna().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_summary = pd.DataFrame({'Missing Count': missing_counts, 'Percentage (%)': missing_pct.round(3)})
print(missing_summary)

print("\n=== Descriptive Statistics for Numeric Columns ===")
df.describe().T

## 4. Visualizing 1 Week of Normal Usage
We examine a typical 7-day period (e.g., January 7–14, 2007) to inspect diurnal patterns (morning & evening peaks vs. nighttime baselines).

In [ ]:
# Select a sample 7-day window
sample_week = df.loc['2007-01-08':'2007-01-14', 'Global_active_power']

fig, ax = plt.subplots(figsize=(15, 6), dpi=120)
ax.plot(sample_week.index, sample_week.values, color='#0284c7', linewidth=1.0, label='Global Active Power (kW)')

ax.set_title('Household Power Consumption (1-Week Normal Sample: Jan 8 – Jan 14, 2007)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Timestamp', fontsize=12)
ax.set_ylabel('Active Power (kW)', fontsize=12)
ax.set_ylim(0, max(sample_week.max() * 1.1, 8))
ax.legend(loc='upper right', frameon=True)
plt.tight_layout()

out_plot_dir = Path('../outputs/plots') if Path('../outputs').exists() else Path('outputs/plots')
out_plot_dir.mkdir(parents=True, exist_ok=True)
plot_filepath = out_plot_dir / 'sample_week_usage.png'
plt.savefig(plot_filepath, dpi=300)
print(f"Sample plot saved to: {plot_filepath}")
plt.show()